# university_syllabus 데이터셋 정제 및 EDA

이 노트북은 `university_syllabus.csv` 원본에서 시작해, AI/소프트웨어/전기전자 계열 추천에 사용할 수 있는 강의계획서 데이터셋을 단계적으로 만드는 과정을 정리한 문서입니다.

진행 순서는 크게 다음과 같습니다.
- 원본 데이터 구조 파악과 기본 EDA
- 1차/2차 정제를 통한 도메인 관련 과목 선별
- 메타데이터 복원과 중복 제거
- 마지막 결측치 기반 품질 정제

산출물은 아래처럼 구분해서 관리합니다.
- 원본 데이터: 현재 폴더의 `university_syllabus.csv`, `university_syllabus_cs.csv`
- 중간 산출물: `outputs/intermediate/`
- 최종 산출물: `outputs/final/`


## 파일 구조 안내

이 노트북은 `원본 -> 중간 정제 결과 -> 최종 품질 정제 결과` 순서로 데이터를 만든다.

- 원본 데이터
  - `university_syllabus.csv`: 전체 강의계획서 원본
  - `university_syllabus_cs.csv`: 별도 파생 CSV(참고용)
- 중간 산출물
  - `outputs/intermediate/01_syllabus_initial_refined.csv`: 1차 정제 결과
  - `outputs/intermediate/02_syllabus_domain_filtered.csv`: 2차 도메인 정제 결과
  - `outputs/intermediate/03_syllabus_deduplicated.csv`: 메타데이터 복원 전 추가 EDA 및 중복 제거 결과
- 최종 산출물
  - `outputs/final/04_syllabus_metadata_restored_deduplicated.csv`: 메타데이터 복원 및 중복 고정 결과
  - `outputs/final/05_syllabus_final_quality_filtered.csv`: 마지막 품질 정제 결과
  - `outputs/final/05_syllabus_removed_by_quality_filter.csv`: 품질 정제에서 제거된 행 검토용 파일

분석 흐름을 따라가면서 각 단계의 목적, 제거 기준, 저장 이유를 함께 기록한다.


## 1. 데이터 로드 및 기본 정보 확인


In [ ]:
# Purpose: 원본 CSV를 불러오고 데이터 크기/스키마/메모리 사용량을 빠르게 점검합니다.
# Input: university_syllabus.csv
# Output: shape_df, schema_df, memory_df, df 샘플 출력

import pandas as pd
from IPython.display import display

# 1. university_syllabus 데이터 로드 및 기본 정보 확인
# 분석의 기준이 되는 원본 CSV를 로드합니다.
df = pd.read_csv("university_syllabus.csv", encoding="utf-8-sig")

# 전체 행/열 개수를 표 형태로 정리합니다.
shape_df = pd.DataFrame({
    "metric": ["rows", "columns"],
    "value": [df.shape[0], df.shape[1]],
})

# 컬럼별 dtype, 결측치, 고유값 수를 함께 요약합니다.
schema_df = pd.DataFrame({
    "column": df.columns,
    "dtype": df.dtypes.astype(str).values,
    "non_null": df.notna().sum().values,
    "null_count": df.isna().sum().values,
    "null_ratio(%)": (df.isna().mean() * 100).round(2).values,
    "nunique": df.nunique(dropna=True).values,
})

# 데이터프레임 전체 메모리 사용량을 KB/MB 단위로 확인합니다.
memory_df = pd.DataFrame({
    "metric": ["memory_usage_kb", "memory_usage_mb"],
    "value": [
        round(df.memory_usage(deep=True).sum() / 1024, 2),
        round(df.memory_usage(deep=True).sum() / 1024 / 1024, 2),
    ],
})

print("[Shape]")
display(shape_df)

print("[Schema]")
display(schema_df)

print("[Memory]")
display(memory_df)

print("[Sample: top 3 rows]")
display(df.head(3))


### 1-1. EDA 시작 템플릿 (요약 통계)
- 아래 표는 수치형/범주형 컬럼을 함께 빠르게 점검하기 위한 기본 요약입니다.
- 기존 `df`를 그대로 사용하며, 이후 결측치/이상치 분석의 기준점으로 활용합니다.


In [ ]:
# Purpose: 전 컬럼에 대한 요약 통계를 한 번에 확인해 이후 정제 기준의 출발점으로 사용합니다.
# Input: 원본 DataFrame df
# Output: summary_df

# 1-1. EDA 시작 템플릿: 전체 요약 통계 테이블
# NOTE: 반드시 이전 셀(데이터 로드 셀) 실행 후 사용하세요. df가 없으면 NameError가 발생합니다.

# 수치형/범주형을 함께 보는 전체 요약 통계를 생성합니다.
summary_df = df.describe(include="all").T

summary_df["missing_count"] = df.isna().sum()
summary_df["missing_ratio(%)"] = (df.isna().mean() * 100).round(2)

# 실무에서 먼저 보고 싶은 지표 순서대로 컬럼을 재배치합니다.
priority_cols = [
    "missing_count", "missing_ratio(%)", "count", "unique", "top", "freq",
    "mean", "std", "min", "25%", "50%", "75%", "max"
]
existing_cols = [c for c in priority_cols if c in summary_df.columns]
remaining_cols = [c for c in summary_df.columns if c not in existing_cols]
summary_df = summary_df[existing_cols + remaining_cols]

summary_df = summary_df.sort_values(by="missing_ratio(%)", ascending=False)

print("[EDA Starter: describe(include='all').T + missing stats]")
display(summary_df)


## 2. 결측치 분석


In [ ]:
# Purpose: 컬럼별 결측치 규모를 정리해 어떤 필드가 풍부하고 어떤 필드가 취약한지 확인합니다.
# Input: 원본 DataFrame df
# Output: missing_df, 결측 컬럼 개수 요약

print("\n" + "=" * 60)
print("2. 결측치 분석")
print("=" * 60)

# 컬럼별 결측 개수와 결측 비율을 계산합니다.
missing = df.isnull().sum()
missing_pct = (df.isnull().sum() / len(df) * 100).round(2)
missing_df = pd.DataFrame({
    "결측수": missing,
    "결측률(%)": missing_pct,
    "비결측수": len(df) - missing
}).sort_values("결측률(%)", ascending=False)

print("\n컬럼별 결측치 현황:")
display(missing_df)

cols_with_missing = missing_df[missing_df["결측수"] > 0]
if len(cols_with_missing) > 0:
    print(f"\n결측치가 있는 컬럼: {len(cols_with_missing)}개")
else:
    print("\n결측치 없음")


### 앞으로의 진행
- `university_syllabus`는 텍스트 기반 강의계획 정보가 많아, 결측치와 함께 텍스트 길이 분포를 같이 확인합니다.
- 정제는 한 번에 끝내지 않고, 목적이 다른 단계로 나누어 진행합니다. 초반에는 불필요한 도메인을 크게 걷어내고, 후반에는 추천 품질을 해칠 수 있는 중복·메타데이터 손실·결측치 문제를 세밀하게 정리합니다.
- 중간 CSV를 남기는 이유는 각 단계의 결과를 비교하고, 어떤 규칙 때문에 행이 제거되었는지 추적하기 위해서입니다.


## 3. 데이터 중복 및 분포 분석


### 3-1. 중복 데이터 확인


In [ ]:
# Purpose: 원본 단계에서 중복 양상을 미리 파악해 이후 중복 제거 전략을 설계합니다.
# Input: 원본 DataFrame df
# Output: 전체 중복 수, business key 기준 중복 수, 샘플 중복 행

print("\n" + "=" * 60)
print("3-1. 중복 데이터 확인")
print("=" * 60)

# 완전 동일 행 수를 확인합니다.
full_duplicates = df.duplicated().sum()
print(f"\n전체 행 중복: {full_duplicates}개")

# 추적용 식별자인 source_row_number 자체의 중복도 점검합니다.
source_row_duplicates = df.duplicated(subset=["source_row_number"]).sum()
print(f"source_row_number 중복: {source_row_duplicates}개")

# 강의 식별에 가까운 business key를 기준으로 중복 여부를 확인합니다.
business_key = ["year", "university_name", "department_name", "course_name", "grade", "semester"]
business_duplicates = df.duplicated(subset=business_key).sum()
print(f"학수 정보 기준 중복(year+university_name+department_name+course_name+grade+semester): {business_duplicates}개")

no_row_id_cols = [col for col in df.columns if col != "source_row_number"]
content_duplicates = df.duplicated(subset=no_row_id_cols).sum()
print(f"source_row_number 제외 완전 중복: {content_duplicates}개")

if content_duplicates > 0:
    dup_rows = df[df.duplicated(subset=no_row_id_cols, keep=False)].sort_values(business_key)
    print("\n중복 샘플:")
    display(dup_rows.head(10))


### 3-2. 범주형 column 분포 분석


In [ ]:
# Purpose: 주요 범주형 컬럼의 분포를 확인해 학과/과목/이수구분 편중 여부를 살핍니다.
# Input: 원본 DataFrame df
# Output: categorical value count tables

print("\n" + "=" * 60)
print("3-2. 범주형 컬럼 분포 분석")
print("=" * 60)

categorical_cols = [
    "year", "university_name", "college_name", "department_name",
    "course_name", "grade", "semester", "course_type"
]

# 상위 빈도값을 확인해 특정 대학/학과/과목으로 쏠림이 있는지 봅니다.
for col in categorical_cols:
    print(f"\n[{col}] 분포 (고유값: {df[col].nunique(dropna=True)}개)")
    vc = df[col].value_counts(dropna=False).head(15)
    vc_df = pd.DataFrame({
        col: vc.index.astype(str),
        "count": vc.values,
        "ratio(%)": (vc.values / len(df) * 100).round(2),
    })
    display(vc_df)


### 3-3. 수치형 column 분포 분석


In [ ]:
# Purpose: 학점/이론/실습 시간 컬럼의 분포를 확인해 비정상 수치가 있는지 점검합니다.
# Input: 원본 DataFrame df
# Output: numeric_summary_df

print("\n" + "=" * 60)
print("3-3. 수치형 컬럼 분포 분석")
print("=" * 60)

numeric_cols = ["credit", "lecture_hours", "practice_hours"]

# 수치형 컬럼별 분포와 0/음수 같은 이상치 후보를 함께 기록합니다.
numeric_summary = []
for col in numeric_cols:
    series = df[col]
    numeric_summary.append({
        "column": col,
        "count": int(series.shape[0]),
        "min": float(series.min()),
        "25%": float(series.quantile(0.25)),
        "50%": float(series.quantile(0.50)),
        "75%": float(series.quantile(0.75)),
        "max": float(series.max()),
        "mean": round(float(series.mean()), 2),
        "zero_count": int((series == 0).sum()),
        "negative_count": int((series < 0).sum()),
    })

numeric_summary_df = pd.DataFrame(numeric_summary)
display(numeric_summary_df)


## 4. 이상치 및 데이터 품질 확인


In [ ]:
# Purpose: 텍스트 길이와 극단값 샘플을 통해 강의목표/교재 정보의 품질을 미리 확인합니다.
# Input: 원본 DataFrame df
# Output: text_length_summary_df, 이상치 샘플

# 4. 이상치 및 데이터 품질 확인
# NOTE: 이전 셀에서 df가 먼저 로드되어 있어야 합니다. df가 없으면 NameError가 발생합니다.

# 강의 설명/교재 관련 텍스트 컬럼의 길이 품질을 점검합니다.
text_cols = [
    "learning_objective", "main_textbook", "sub_textbook",
    "reference_material", "prerequisite_material"
]

text_length_summary = []
quality_samples = {}

# 각 텍스트 컬럼에 대해 길이 분포와 극단값 샘플을 수집합니다.
for col in text_cols:
    text_len = df[col].fillna("").astype(str).str.len()
    text_length_summary.append({
        "column": col,
        "count": int(text_len.shape[0]),
        "min": int(text_len.min()),
        "25%": round(float(text_len.quantile(0.25)), 2),
        "50%": round(float(text_len.quantile(0.50)), 2),
        "75%": round(float(text_len.quantile(0.75)), 2),
        "max": int(text_len.max()),
        "mean": round(float(text_len.mean()), 2),
        "len<=2": int((text_len <= 2).sum()),
        "len>=500": int((text_len >= 500).sum()),
    })

    short_mask = text_len <= 2
    long_mask = text_len >= 500

    quality_samples[col] = {
        "short": df.loc[short_mask, ["course_name", col]].head(5),
        "long": df.loc[long_mask, ["course_name", col]].head(5),
    }

text_length_summary_df = pd.DataFrame(text_length_summary)
print("[text column length summary]")
display(text_length_summary_df)

zero_credit_df = df.loc[df["credit"] == 0, ["university_name", "department_name", "course_name", "credit"]]
high_practice_df = df.loc[df["practice_hours"] >= 10, [
    "university_name", "department_name", "course_name", "practice_hours"
]].sort_values("practice_hours", ascending=False)

print("\n[credit == 0 sample]")
display(zero_credit_df.head(10))

print("\n[practice_hours >= 10 sample]")
display(high_practice_df.head(10))

# 각 텍스트 컬럼에 대해 길이 분포와 극단값 샘플을 수집합니다.
for col in text_cols:
    print(f"\n[{col}] short sample")
    display(quality_samples[col]["short"])
    print(f"[{col}] long sample")
    display(quality_samples[col]["long"])


### 이상치 및 데이터 품질 분석 결과
- `practice_hours`는 일부 과목에서 큰 값이 나타나므로, 삭제 전 샘플 확인이 필요합니다.
- 텍스트 컬럼은 실제 결측치 외에도 `없음` 계열 문자열이 섞여 있어, 정제 시 결측으로 통일하는 것이 분석에 유리합니다.


## 5. 정제 작업 수행

이 단계는 도메인 관련 과목만 남기기 위한 2차 정제의 시작점입니다. 여기서는 `department_name`과 `course_name`의 양성 키워드를 사용해 컴퓨터공학, AI, 데이터, 전기전자, 반도체, 제어, 로봇 계열 후보를 우선 남기고, 이후 `course_type`과 노이즈 키워드로 거친 정제를 수행합니다.

중요한 의도는 '무엇을 제거할지'보다 '무엇을 추천 후보로 남길지'를 먼저 정의하는 것입니다.


In [ ]:
# Purpose: AI/CS/EE 관련 과목만 남기기 위한 2차 양성 필터링을 수행합니다.
# Input: 원본 DataFrame df
# Output: df_cs_ee_clean, cleaning_log_df, 검토용 ambiguous_review_df

# 5. 2차 정제 작업 수행
# NOTE: 이전 셀에서 df가 먼저 로드되어 있어야 합니다. df가 없으면 NameError가 발생합니다.

from pathlib import Path
import re

# 원본 df는 유지하고, 정제 전용 작업용 복사본에서만 필터링을 진행합니다.
df_work = df.copy()

rows_before = len(df_work)
cols_before = df_work.shape[1]
# 단계별 행/열 변화량을 기록해 어떤 규칙이 얼마나 영향을 줬는지 추적합니다.
cleaning_logs = []
removed_frames = []

def add_cleaning_log(step, rows_before_step, rows_after_step, cols_before_step, cols_after_step, note):
    """기준별 행/열 변화량을 기록해 정제 과정을 재현 가능하게 남깁니다."""
    cleaning_logs.append({
        "step": step,
        "rows_before": rows_before_step,
        "rows_after": rows_after_step,
        "rows_removed": rows_before_step - rows_after_step,
        "cols_before": cols_before_step,
        "cols_after": cols_after_step,
        "cols_removed": cols_before_step - cols_after_step,
        "note": note,
    })

# 학과명이 목표 도메인과 관련 있는지 판단하는 양성 키워드입니다.
department_keywords = [
    "컴퓨터", "소프트웨어", "인공지능", "AI", "데이터", "정보", "전산", "전기", "전자",
    "정보통신", "통신", "반도체", "제어", "로봇", "임베디드", "사이버", "보안",
    "IoT", "지능", "융합소프트웨어", "스마트", "디지털"
]
# 과목명만 보고도 도메인 관련성을 잡아낼 수 있도록 핵심 과목 키워드를 정의합니다.
course_keywords = [
    "프로그래밍", "코딩", "파이썬", "Python", "자바", "Java", "C언어", "C++", "객체지향",
    "자료구조", "알고리즘", "운영체제", "데이터베이스", "DB", "컴퓨터구조", "컴퓨터아키텍처",
    "네트워크", "소프트웨어공학", "웹", "앱", "모바일", "서버", "클라우드", "시스템프로그래밍",
    "오픈소스", "캡스톤", "프로젝트", "인공지능", "AI", "머신러닝", "기계학습", "딥러닝",
    "빅데이터", "데이터사이언스", "데이터분석", "데이터마이닝", "자연어처리", "NLP",
    "컴퓨터비전", "영상처리", "패턴인식", "추천시스템", "강화학습", "통계", "확률", "선형대수",
    "이산수학", "수치해석", "최적화", "전기", "전자", "회로", "논리회로", "디지털논리",
    "전자회로", "전기회로", "회로이론", "전자기학", "신호", "신호처리", "통신", "정보통신",
    "무선통신", "반도체", "마이크로프로세서", "마이크로컨트롤러", "임베디드", "센서",
    "제어", "로봇", "IoT", "디지털시스템", "VLSI", "FPGA", "보안", "정보보호", "암호",
    "해킹", "사이버", "네트워크보안"
]
noise_# 과목명만 보고도 도메인 관련성을 잡아낼 수 있도록 핵심 과목 키워드를 정의합니다.
course_keywords = [
    "진로", "진로설계", "취업", "창업", "실용영어", "영어", "의사소통", "글쓰기", "발표",
    "인성", "봉사", "리더십", "체육", "예술", "디자인", "교양", "꿈미래", "개척"
]
broad_department_keywords = ["정보", "스마트", "디지털"]
ambiguous_course_keywords = ["정보", "스마트", "디지털", "실험"]
low_relevance_experiment_keywords = ["일반물리실험", "일반화학실험"]

# contains 검색에 사용할 정규식 패턴을 미리 조립합니다.
department_pattern = "|".join(re.escape(keyword) for keyword in department_keywords)
course_pattern = "|".join(re.escape(keyword) for keyword in course_keywords)
noise_course_pattern = "|".join(re.escape(keyword) for keyword in noise_course_keywords)
broad_department_pattern = "|".join(re.escape(keyword) for keyword in broad_department_keywords)
ambiguous_course_pattern = "|".join(re.escape(keyword) for keyword in ambiguous_course_keywords)
low_relevance_experiment_pattern = "|".join(re.escape(keyword) for keyword in low_relevance_experiment_keywords)

# 문자열 비교 오차를 줄이기 위해 결측치와 공백을 먼저 정리합니다.
for col in ["department_name", "course_name", "course_type"]:
    df_work[col] = df_work[col].fillna("").astype(str).str.strip()

# 학과명과 과목명 기준으로 각각 타깃 여부 플래그를 만듭니다.
df_work["is_target_department"] = df_work["department_name"].str.contains(
    department_pattern, case=False, regex=True, na=False
)
df_work["is_target_course"] = df_work["course_name"].str.contains(
    course_pattern, case=False, regex=True, na=False
)
df_work["is_broad_department_match"] = df_work["department_name"].str.contains(
    broad_department_pattern, case=False, regex=True, na=False
)
df_work["is_ambiguous_course"] = df_work["course_name"].str.contains(
    ambiguous_course_pattern, case=False, regex=True, na=False
)
df_work["is_low_relevance_experiment"] = df_work["course_name"].str.contains(
    low_relevance_experiment_pattern, case=False, regex=True, na=False
)

rows_before_step = len(df_work)
cols_before_step = df_work.shape[1]
# 이번 단계의 핵심: 학과 또는 과목명 둘 중 하나라도 타깃이면 우선 유지합니다.
positive_mask = df_work["is_target_department"] | df_work["is_target_course"]
positive_kept_rows = int(positive_mask.sum())
positive_removed_rows = int((~positive_mask).sum())
removed_positive_df = df_work.loc[~positive_mask].copy()
removed_positive_df["removal_reason"] = "not_matched_by_positive_filter"
removed_frames.append(removed_positive_df)
df_filtered = df_work.loc[positive_mask].copy()
add_cleaning_log(
    step="positive_filter_keep_target_department_or_course",
    rows_before_step=rows_before_step,
    rows_after_step=len(df_filtered),
    cols_before_step=cols_before_step,
    cols_after_step=df_filtered.shape[1],
    note="keep rows where is_target_department OR is_target_course is True",
)

rows_before_step = len(df_filtered)
cols_before_step = df_filtered.shape[1]
# 양성 필터를 통과했더라도 일반 교양/진로성 과목명은 다시 제거합니다.
noise_mask = df_filtered["course_name"].str.contains(noise_course_pattern, case=False, regex=True, na=False)
noise_removed_rows = int(noise_mask.sum())
removed_noise_df = df_filtered.loc[noise_mask].copy()
removed_noise_df["removal_reason"] = "noise_course_keyword"
removed_frames.append(removed_noise_df)
df_filtered = df_filtered.loc[~noise_mask].copy()
add_cleaning_log(
    step="remove_noise_course_name",
    rows_before_step=rows_before_step,
    rows_after_step=len(df_filtered),
    cols_before_step=cols_before_step,
    cols_after_step=df_filtered.shape[1],
    note="remove rows where course_name contains noise keywords",
)

rows_before_step = len(df_filtered)
cols_before_step = df_filtered.shape[1]
# 실제 이수구분이 교양인 경우 전공 추천셋 목적에 맞지 않아 제외합니다.
course_type_mask = df_filtered["course_type"].str.contains("교양", case=False, regex=True, na=False)
course_type_removed_rows = int(course_type_mask.sum())
removed_course_type_df = df_filtered.loc[course_type_mask].copy()
removed_course_type_df["removal_reason"] = "course_type_contains_교양"
removed_frames.append(removed_course_type_df)
df_filtered = df_filtered.loc[~course_type_mask].copy()
add_cleaning_log(
    step="filter_course_type",
    rows_before_step=rows_before_step,
    rows_after_step=len(df_filtered),
    cols_before_step=cols_before_step,
    cols_after_step=df_filtered.shape[1],
    note="course_type contains '교양'",
)

retained_course_counts = (
    df_filtered["course_name"]
    .value_counts(dropna=False)
    .rename_axis("course_name")
    .reset_index(name="count")
    .head(100)
)
retained_course_sample = retained_course_counts["course_name"].tolist()

removed_all_df = pd.concat(removed_frames, ignore_index=True) if removed_frames else pd.DataFrame(columns=df_work.columns.tolist() + ["removal_reason"])
removed_course_counts = (
    removed_all_df["course_name"]
    .value_counts(dropna=False)
    .rename_axis("course_name")
    .reset_index(name="count")
    .head(100)
)
removed_course_sample = removed_course_counts["course_name"].tolist()

# 정보/스마트/디지털/실험처럼 해석이 애매한 과목은 별도 검토용 목록으로 남깁니다.
ambiguous_review_df = df_filtered.loc[
    df_filtered["is_broad_department_match"]
    | df_filtered["is_ambiguous_course"]
    | df_filtered["is_low_relevance_experiment"],
    [
        "source_row_number", "department_name", "course_name", "course_type", "credit",
        "is_target_department", "is_target_course", "is_broad_department_match",
        "is_ambiguous_course", "is_low_relevance_experiment"
    ]
].drop_duplicates().sort_values(["department_name", "course_name"]).reset_index(drop=True)

# 행 필터링이 끝난 뒤에만 불필요 컬럼을 제거해 이후 병합/검증과 충돌하지 않게 합니다.
columns_to_drop = [
    "year",
    "university_name",
    "college_name",
    "department_name",
    "lecture_hours",
    "practice_hours",
]
existing_columns_to_drop = [col for col in columns_to_drop if col in df_filtered.columns]
helper_# 행 필터링이 끝난 뒤에만 불필요 컬럼을 제거해 이후 병합/검증과 충돌하지 않게 합니다.
columns_to_drop = [
    "is_broad_department_match",
    "is_ambiguous_course",
    "is_low_relevance_experiment",
]
existing_helper_columns_to_drop = [col for col in helper_columns_to_drop if col in df_filtered.columns]

rows_before_step = len(df_filtered)
cols_before_step = df_filtered.shape[1]
df_cs_ee_clean = df_filtered.drop(columns=existing_columns_to_drop + existing_helper_columns_to_drop).copy()
add_cleaning_log(
    step="drop_columns",
    rows_before_step=rows_before_step,
    rows_after_step=len(df_cs_ee_clean),
    cols_before_step=cols_before_step,
    cols_after_step=df_cs_ee_clean.shape[1],
    note=f"dropped existing columns: {existing_columns_to_drop + existing_helper_columns_to_drop}",
)

rows_after = len(df_cs_ee_clean)
cols_after = df_cs_ee_clean.shape[1]
total_removed_rows = rows_before - rows_after
total_removed_ratio = round((total_removed_rows / rows_before) * 100, 2) if rows_before else 0.0

cleaning_log_df = pd.DataFrame(cleaning_logs)

# 이후 단계에서 재사용할 수 있도록 2차 정제 결과를 intermediate 산출물로 저장합니다.
output_path = Path("outputs/intermediate/02_syllabus_domain_filtered.csv")
df_cs_ee_clean.to_csv(output_path, index=False, encoding="utf-8-sig")

print("[1] 정제 전 행/열 개수")
print(f"rows: {rows_before}, cols: {cols_before}")

print("\n[2] 양성 필터링으로 유지된 행 수")
print(positive_kept_rows)

print("\n[3] 양성 필터링으로 제거된 행 수")
print(positive_removed_rows)

print("\n[4] 노이즈 course_name 키워드로 제거된 행 수")
print(noise_removed_rows)

print("\n[5] course_type의 '교양' 기준으로 제거된 행 수")
print(course_type_removed_rows)

print("\n[6] 최종 행/열 개수")
print(f"rows: {rows_after}, cols: {cols_after}")

print("\n[7] 전체 제거 행 수")
print(total_removed_rows)

print("\n[8] 전체 제거 비율")
print(f"{total_removed_ratio}%")

print("\n[9] 단계별 정제 로그 DataFrame")
display(cleaning_log_df)

print("\n[10] 유지된 과목 중 course_name 고유값 상위 100개")
display(retained_course_counts)

print("\n[11] 제거된 과목 중 course_name 고유값 상위 100개")
display(removed_course_counts)

print("\n[12] 애매한 과목 검토용 DataFrame")
display(ambiguous_review_df)

print(f"\n[Saved] {output_path.resolve()}")


In [ ]:
# Purpose: 2차 정제 결과를 한 번에 검토할 수 있도록 핵심 출력만 모아서 보여줍니다.

display(cleaning_log_df)
display(df_cs_ee_clean.head(5))
display(retained_course_sample)
display(removed_course_sample)
display(ambiguous_review_df)


## 6. 정제 결과 검증


In [ ]:
# Purpose: 2차 정제 이후 남은 데이터 규모와 결측 상태를 검증합니다.
# Output: verification_df, null summary after cleaning

# 2차 정제 이후 핵심 집계값을 표로 다시 확인합니다.
verification_df = pd.DataFrame({
    "metric": [
        "remaining_rows",
        "remaining_columns",
        "positive_kept_rows",
        "noise_removed_rows",
        "ambiguous_review_rows",
    ],
    "value": [
        len(df_cs_ee_clean),
        df_cs_ee_clean.shape[1],
        positive_kept_rows,
        noise_removed_rows,
        len(ambiguous_review_df),
    ],
})

print("[Verification]")
display(verification_df)

print("[Null summary after cleaning]")
display(
    pd.DataFrame({
        "column": df_cs_ee_clean.columns,
        "null_count": df_cs_ee_clean.isna().sum().values,
        "null_ratio(%)": (df_cs_ee_clean.isna().mean() * 100).round(2).values,
    }).sort_values("null_ratio(%)", ascending=False)
)


## 7. 메타데이터 복원, 추가 EDA 및 중복 제거

2차 정제 결과만 보면 과목 자체는 남아 있지만, 어떤 단과대학/학과에서 온 과목인지 해석하기 어려울 수 있습니다. 이 단계에서는 `source_row_number`를 기준으로 원본의 `college_name`, `department_name`을 다시 붙여서 도메인 적합성과 잔여 노이즈를 확인합니다.

동시에 같은 강의계획서 내용이 반복 등록된 경우를 찾아내어, 이후 추천 데이터셋이 특정 과목에 과도하게 편향되지 않도록 중복 제거를 수행합니다.


In [ ]:
# Purpose: source_row_number를 이용해 메타데이터를 복원하고, 추가 EDA 및 1차 중복 제거를 수행합니다.
# Input: 원본 df, 2차 정제 결과 df_cs_ee_clean
# Output: df_final_eda, route_summary_df, 중복 분석 결과

# 7. 메타데이터 복원, 추가 EDA 및 중복 제거
# NOTE: df, df_cs_ee_clean 이 먼저 생성되어 있어야 합니다.

from pathlib import Path
import re

def normalize_text(value):
    """중복 비교 전에 공백 차이와 결측성 텍스트를 최대한 일관되게 정리합니다."""
    if pd.isna(value):
        return ""
    text = str(value).strip()
    text = re.sub(r"\s+", " ", text)
    return text

eda_logs = []

def add_eda_log(step, rows_before, rows_after, cols_before, cols_after, note):
    """추가 EDA/중복 제거 단계에서 행·열 변화를 기록합니다."""
    eda_logs.append({
        "step": step,
        "rows_before": rows_before,
        "rows_after": rows_after,
        "rows_removed": rows_before - rows_after,
        "cols_before": cols_before,
        "cols_after": cols_after,
        "cols_removed": cols_before - cols_after,
        "note": note,
    })

# 원본과 정제본을 분리해, 메타데이터 복원 과정에서 원본이 훼손되지 않게 합니다.
df_original = df.copy()
df_target = df_cs_ee_clean.copy()

rows_before_restore = len(df_target)
cols_before_restore = df_target.shape[1]

# 원본에서 메타데이터 복원에 필요한 키와 학과 정보를 따로 준비합니다.
source_meta_df = df_original.reset_index().rename(columns={"index": "original_index"}).copy()
source_meta_df["source_row_number"] = source_meta_df["source_row_number"].astype(str).map(normalize_text)
df_target["source_row_number"] = df_target["source_row_number"].astype(str).map(normalize_text)

merge_meta_df = source_meta_df[[
    "source_row_number", "original_index", "college_name", "department_name"
]].copy()
merge_meta_df = merge_meta_df.drop_duplicates(subset=["source_row_number"], keep="first")

# source_row_number 기준으로 college_name, department_name을 다시 붙입니다.
df_work = df_target.merge(
    merge_meta_df,
    on="source_row_number",
    how="left",
    validate="many_to_one"
)

college_missing_count = int(df_work["college_name"].isna().sum()) if "college_name" in df_work.columns else len(df_work)
department_missing_count = int(df_work["department_name"].isna().sum()) if "department_name" in df_work.columns else len(df_work)
merge_failed_sample_df = df_work.loc[
    df_work.get("college_name", pd.Series(index=df_work.index, dtype=object)).isna()
    | df_work.get("department_name", pd.Series(index=df_work.index, dtype=object)).isna()
].head(20).copy()

print("[Restore] 병합 전 df_cs_ee_clean 행 수")
print(rows_before_restore)
print("\n[Restore] 병합 후 df_work 행 수")
print(len(df_work))
print("\n[Restore] college_name 결측치 수")
print(college_missing_count)
print("\n[Restore] department_name 결측치 수")
print(department_missing_count)
print("\n[Restore] 병합 실패 의심 행 샘플")
display(merge_failed_sample_df)

add_eda_log(
    "restore_college_department",
    rows_before_restore,
    len(df_work),
    cols_before_restore,
    df_work.shape[1],
    f"merged college_name/department_name by source_row_number; missing college={college_missing_count}, missing department={department_missing_count}",
)

for col in df_work.columns:
    if df_work[col].dtype == "object":
        df_work[col] = df_work[col].map(normalize_text)

# 어떤 경로(학과 기준/과목명 기준)로 살아남았는지 4가지 경우로 분해합니다.
route_records = [
    {
        "route": "department=True, course=True",
        "is_target_department": True,
        "is_target_course": True,
        "count": int(((df_work["is_target_department"] == True) & (df_work["is_target_course"] == True)).sum()),
    },
    {
        "route": "department=True, course=False",
        "is_target_department": True,
        "is_target_course": False,
        "count": int(((df_work["is_target_department"] == True) & (df_work["is_target_course"] == False)).sum()),
    },
    {
        "route": "department=False, course=True",
        "is_target_department": False,
        "is_target_course": True,
        "count": int(((df_work["is_target_department"] == False) & (df_work["is_target_course"] == True)).sum()),
    },
    {
        "route": "department=False, course=False",
        "is_target_department": False,
        "is_target_course": False,
        "count": int(((df_work["is_target_department"] == False) & (df_work["is_target_course"] == False)).sum()),
    },
]
route_summary_df = pd.DataFrame(route_records)

route_false_true_df = df_work.loc[
    (df_work["is_target_department"] == False) & (df_work["is_target_course"] == True)
].copy()
route_false_true_sample_df = route_false_true_df.head(50)
route_false_true_department_df = (
    route_false_true_df["department_name"]
    .value_counts(dropna=False)
    .rename_axis("department_name")
    .reset_index(name="count")
    .head(50)
)
route_false_true_course_df = (
    route_false_true_df["course_name"]
    .value_counts(dropna=False)
    .rename_axis("course_name")
    .reset_index(name="count")
    .head(50)
)

display(route_summary_df)
display(route_false_true_sample_df)
display(route_false_true_department_df)
display(route_false_true_course_df)

add_eda_log(
    "route_analysis",
    len(df_work),
    len(df_work),
    df_work.shape[1],
    df_work.shape[1],
    f"false-true route rows={len(route_false_true_df)}",
)

# 남은 데이터가 특정 단과대학/학과에 치우쳐 있는지 분포를 확인합니다.
college_count_df = (
    df_work["college_name"]
    .value_counts(dropna=False)
    .rename_axis("college_name")
    .reset_index(name="count")
)
department_count_df = (
    df_work["department_name"]
    .value_counts(dropna=False)
    .rename_axis("department_name")
    .reset_index(name="count")
)
college_department_count_df = (
    df_work.groupby(["college_name", "department_name"], dropna=False)
    .size()
    .reset_index(name="count")
    .sort_values("count", ascending=False)
)

display(college_count_df)
display(department_count_df.head(100))
display(college_department_count_df.head(100))

add_eda_log(
    "department_distribution",
    len(df_work),
    len(df_work),
    df_work.shape[1],
    df_work.shape[1],
    f"unique colleges={college_count_df.shape[0]}, unique departments={department_count_df.shape[0]}",
)

# 도메인과 거리가 있을 수 있는 학과 키워드를 review 후보로만 따로 모읍니다.
noise_department_keywords = [
    "건축", "토목", "도시", "기계", "화학", "환경", "생명", "식품", "의류", "경영", "경제", "행정",
    "법", "인문", "사회", "교육", "예술", "체육", "간호", "의학", "약학", "농업", "산림"
]
noise_department_pattern = "|".join(re.escape(keyword) for keyword in noise_department_keywords)
noise_department_mask = (
    df_work["department_name"].str.contains(noise_department_pattern, regex=True, na=False)
    | df_work["college_name"].str.contains(noise_department_pattern, regex=True, na=False)
)
review_noise_department_df = df_work.loc[noise_department_mask].copy()
review_noise_department_sample_df = review_noise_department_df.head(100)
review_noise_department_summary_df = (
    review_noise_department_df["department_name"]
    .value_counts(dropna=False)
    .rename_axis("department_name")
    .reset_index(name="count")
)
review_noise_course_summary_df = (
    review_noise_department_df["course_name"]
    .value_counts(dropna=False)
    .rename_axis("course_name")
    .reset_index(name="count")
)

display(review_noise_department_sample_df)
display(review_noise_department_summary_df)
display(review_noise_course_summary_df)

add_eda_log(
    "noise_department_review",
    len(df_work),
    len(df_work),
    df_work.shape[1],
    df_work.shape[1],
    f"noise review candidate rows={len(review_noise_department_df)}",
)

# 중복 탐색은 복원된 메타데이터를 포함한 작업본 기준으로 수행합니다.
comparison_df = df_work.copy()
comparison_cols = comparison_df.columns.tolist()
comparison_subset_exact = [
    col for col in comparison_cols if col not in ["source_row_number", "original_index"]
]

# source_row_number만 다른 완전 중복 행이 있는지 먼저 확인합니다.
exact_duplicate_mask = comparison_df.duplicated(subset=comparison_subset_exact, keep=False)
exact_duplicate_count = int(comparison_df.duplicated(subset=comparison_subset_exact).sum())
exact_duplicate_sample_df = comparison_df.loc[exact_duplicate_mask].sort_values(comparison_subset_exact[:3]).head(50)

display(exact_duplicate_sample_df)

add_eda_log(
    "exact_duplicate_check",
    len(df_work),
    len(df_work),
    df_work.shape[1],
    df_work.shape[1],
    f"exact duplicate rows={exact_duplicate_count}",
)

# 완전 중복보다 느슨하게, 강의계획서 내용이 사실상 같은 경우도 중복 후보로 봅니다.
syllabus_duplicate_cols = [
    "course_name", "grade", "semester", "credit", "course_type", "learning_objective",
    "main_textbook", "sub_textbook", "reference_material", "prerequisite_material",
    "college_name", "department_name"
]
syllabus_duplicate_cols = [col for col in syllabus_duplicate_cols if col in comparison_df.columns]
syllabus_duplicate_mask = comparison_df.duplicated(subset=syllabus_duplicate_cols, keep=False)
syllabus_duplicate_count = int(comparison_df.duplicated(subset=syllabus_duplicate_cols).sum())
syllabus_duplicate_sample_df = comparison_df.loc[syllabus_duplicate_mask].sort_values(syllabus_duplicate_cols[:3]).head(50)

# course_name 기준 중복은 제거하지 않고, 대표 과목명 통합 후보 분석용으로만 남깁니다.
course_name_count_df = (
    comparison_df["course_name"]
    .value_counts(dropna=False)
    .rename_axis("course_name")
    .reset_index(name="count")
)

display(syllabus_duplicate_sample_df)
display(course_name_count_df.head(100))

add_eda_log(
    "syllabus_duplicate_check",
    len(df_work),
    len(df_work),
    df_work.shape[1],
    df_work.shape[1],
    f"syllabus duplicate rows={syllabus_duplicate_count}; course_name duplicates tracked separately",
)

rows_before_exact_drop = len(comparison_df)
cols_before_exact_drop = comparison_df.shape[1]
# 1차로 완전 중복을 제거한 뒤, 2차로 강의계획서 내용 기준 중복을 제거합니다.
df_dedup = comparison_df.drop_duplicates(subset=comparison_subset_exact, keep="first").copy()
rows_after_exact_drop = len(df_dedup)
add_eda_log(
    "exact_duplicate_drop",
    rows_before_exact_drop,
    rows_after_exact_drop,
    cols_before_exact_drop,
    df_dedup.shape[1],
    "drop duplicates excluding source_row_number",
)

rows_before_syllabus_drop = len(df_dedup)
cols_before_syllabus_drop = df_dedup.shape[1]
df_dedup = df_dedup.drop_duplicates(subset=syllabus_duplicate_cols, keep="first").copy()
rows_after_syllabus_drop = len(df_dedup)
add_eda_log(
    "syllabus_duplicate_drop",
    rows_before_syllabus_drop,
    rows_after_syllabus_drop,
    cols_before_syllabus_drop,
    df_dedup.shape[1],
    f"drop duplicates on syllabus key columns: {syllabus_duplicate_cols}",
)

course_name_department_count_df = (
    df_dedup.groupby("course_name", dropna=False)["department_name"]
    .nunique()
    .reset_index(name="unique_department_count")
    .sort_values("unique_department_count", ascending=False)
)
course_name_objective_count_df = (
    df_dedup.assign(_objective_norm=df_dedup["learning_objective"].map(normalize_text))
    .groupby("course_name", dropna=False)["_objective_norm"]
    .nunique()
    .reset_index(name="unique_learning_objective_count")
    .sort_values("unique_learning_objective_count", ascending=False)
)
same_course_diff_objective_df = (
    df_dedup.assign(_objective_norm=df_dedup["learning_objective"].map(normalize_text))
    .groupby("course_name")
    .filter(lambda x: x["_objective_norm"].nunique() > 1)
    .drop(columns=["_objective_norm"])
    .sort_values(["course_name", "department_name", "grade", "semester"])
)

display(course_name_count_df.head(100))
display(course_name_department_count_df.head(100))
display(course_name_objective_count_df.head(100))
display(same_course_diff_objective_df.head(100))

df_final_eda = df_dedup.drop(columns=["original_index"], errors="ignore").copy()
# 추가 EDA와 1차 중복 제거 결과를 다음 단계 입력용 intermediate 파일로 저장합니다.
output_path = Path("outputs/intermediate/03_syllabus_deduplicated.csv")
df_final_eda.to_csv(output_path, index=False, encoding="utf-8-sig")

add_eda_log(
    "save_final_dataset",
    len(df_final_eda),
    len(df_final_eda),
    df_final_eda.shape[1],
    df_final_eda.shape[1],
    f"saved final dataset to {output_path.name}",
)

eda_log_df = pd.DataFrame(eda_logs)

print("[Duplicate] 완전 중복 행 수")
print(exact_duplicate_count)
print("\n[Duplicate] 강의계획서 기준 중복 행 수")
print(syllabus_duplicate_count)
print("\n[Duplicate] 중복 제거 전 행 수")
print(rows_before_exact_drop)
print("\n[Duplicate] 완전 중복 제거 후 행 수")
print(rows_after_exact_drop)
print("\n[Duplicate] 강의계획서 기준 중복 제거 후 행 수")
print(rows_after_syllabus_drop)
print("\n[Duplicate] 최종 제거 행 수")
print(rows_before_exact_drop - rows_after_syllabus_drop)
print("\n[Duplicate] 최종 제거 비율")
print(f"{round(((rows_before_exact_drop - rows_after_syllabus_drop) / rows_before_exact_drop) * 100, 2) if rows_before_exact_drop else 0.0}%")

display(eda_log_df)
display(route_summary_df)
display(college_count_df)
display(department_count_df.head(100))
display(review_noise_department_summary_df)
display(exact_duplicate_sample_df)
display(syllabus_duplicate_sample_df)
display(course_name_count_df.head(100))
display(df_final_eda.head(5))

print(f"[Saved] {output_path.resolve()}")


## 8. 후처리 재실행: 메타데이터 복원 및 중복 고정

이 단계는 7번 섹션의 분석 결과를 바탕으로, 실제 최종 추천용 데이터셋을 만들기 위해 중복 제거 기준을 더 엄격하게 다시 적용하는 단계입니다.

핵심 원칙은 다음과 같습니다.
- `source_row_number`는 추적용이므로 중복 판단 기준에서 제외
- `college_name`, `department_name`은 결과 해석에는 필요하지만, 강의 내용 중복 여부 판단에서는 제외
- 즉, 같은 강의계획서 내용이면 메타데이터가 조금 달라도 하나의 행으로 정리


In [ ]:
# Purpose: 메타데이터 복원과 중복 제거를 더 엄격한 기준으로 다시 실행해 안정적인 추천용 기반 파일을 만듭니다.
# Input: df 또는 저장된 중간 산출물
# Output: df_final_fixed, eda_fix_log_df

# 8. 후처리 재실행: 메타데이터 복원 및 중복 고정
# NOTE: 원본 df는 수정하지 않고 copy()로만 사용합니다.

from pathlib import Path
import re
import html

eda_fix_logs = []

def add_fix_log(step, rows_before, rows_after, cols_before, cols_after, note):
    """후처리 재실행 단계의 핵심 변경 사항을 로그 테이블로 남깁니다."""
    eda_fix_logs.append({
        "step": step,
        "rows_before": rows_before,
        "rows_after": rows_after,
        "rows_removed": rows_before - rows_after,
        "cols_before": cols_before,
        "cols_after": cols_after,
        "cols_removed": cols_before - cols_after,
        "note": note,
    })

def normalize_text(value):
    """중복 비교 전에 공백 차이와 결측성 텍스트를 최대한 일관되게 정리합니다."""
    if pd.isna(value):
        return ""
    text = html.unescape(str(value))
    text = text.replace("\u3000", " ")
    text = text.replace("\xa0", " ")
    text = text.replace("\u200b", " ")
    text = text.replace("\ufeff", " ")
    text = text.replace("\r", " ").replace("\n", " ").replace("\t", " ")
    text = re.sub(r"\s+", " ", text).strip()
    if text in {"", "없음", "없음.", "해당없음", "해당 없음", "없다", "-"}:
        return "없음"
    return text

df_original = df.copy()

base_dir = Path(".")
# 메모리에 정제본이 없으면 직전 intermediate 산출물에서 이어서 작업할 수 있게 합니다.
candidate_files = [
    base_dir / "outputs/intermediate/03_syllabus_deduplicated.csv",
    base_dir / "outputs/intermediate/02_syllabus_domain_filtered.csv",
]

if "df_cs_ee_clean" in globals():
    df_clean_source = df_cs_ee_clean.copy()
    clean_data_source_note = "used in-memory df_cs_ee_clean"
else:
    existing_candidate = next((path for path in candidate_files if path.exists()), None)
    if existing_candidate is None:
        raise FileNotFoundError("df_cs_ee_clean이 없고, outputs/intermediate/03_syllabus_deduplicated.csv 또는 outputs/intermediate/02_syllabus_domain_filtered.csv도 찾을 수 없습니다.")
    df_clean_source = pd.read_csv(existing_candidate)
    clean_data_source_note = f"loaded from {existing_candidate.name}"

add_fix_log(
    "load_or_prepare_clean_data",
    len(df_clean_source),
    len(df_clean_source),
    df_clean_source.shape[1],
    df_clean_source.shape[1],
    clean_data_source_note,
)

required_meta_cols = ["college_name", "department_name"]
missing_meta_cols = [col for col in required_meta_cols if col not in df_original.columns]
if missing_meta_cols:
    message = f"원본 df에서 메타데이터 컬럼을 찾을 수 없습니다: {missing_meta_cols}"
    print(message)
    raise ValueError(message)

# 원본 df에서 source_row_number와 메타데이터만 따로 추출해 merge 준비를 합니다.
if "source_row_number" in df_original.columns:
    df_meta = df_original[["source_row_number", "college_name", "department_name"]].copy()
else:
    df_meta = df_original.reset_index().rename(columns={"index": "source_row_number"})[["source_row_number", "college_name", "department_name"]].copy()

rows_before_restore = len(df_clean_source)
cols_before_restore = df_clean_source.shape[1]

df_clean_prepare = df_clean_source.copy()
# 기존에 남아 있던 메타데이터 컬럼은 제거하고 원본 기준으로 다시 복원합니다.
for meta_col in ["college_name", "department_name"]:
    if meta_col in df_clean_prepare.columns:
        df_clean_prepare = df_clean_prepare.drop(columns=[meta_col])

# merge key 타입을 맞춰 source_row_number 기준 병합 오류를 줄입니다.
df_meta["source_row_number"] = pd.to_numeric(df_meta["source_row_number"], errors="coerce")
df_clean_prepare["source_row_number"] = pd.to_numeric(df_clean_prepare["source_row_number"], errors="coerce")

df_work = df_clean_prepare.merge(df_meta, on="source_row_number", how="left", validate="many_to_one")

rows_after_restore = len(df_work)
college_missing_count = int(df_work["college_name"].isna().sum())
department_missing_count = int(df_work["department_name"].isna().sum())
merge_failed_sample_df = df_work.loc[
    df_work["college_name"].isna() | df_work["department_name"].isna()
].head(20).copy()

print("[Restore] 병합 전 행 수")
print(rows_before_restore)
print("\n[Restore] 병합 후 행 수")
print(rows_after_restore)
if rows_before_restore != rows_after_restore:
    print("WARNING: merge 후 행 수가 변경되었습니다. source_row_number 중복 또는 merge 문제를 확인하세요.")
print("\n[Restore] college_name 결측치 수")
print(college_missing_count)
print("\n[Restore] department_name 결측치 수")
print(department_missing_count)
print("\n[Restore] source_row_number 기준 merge 실패 의심 행 샘플")
display(merge_failed_sample_df)
print("\n[Restore] 복원된 df_work.head()")
display(df_work.head())

add_fix_log(
    "restore_college_department",
    rows_before_restore,
    rows_after_restore,
    cols_before_restore,
    df_work.shape[1],
    f"restored college_name/department_name by source_row_number; missing college={college_missing_count}, missing department={department_missing_count}",
)

rows_before_norm = len(df_work)
cols_before_norm = df_work.shape[1]
df_norm = df_work.copy()
# 중복 비교 전에 문자열 컬럼을 정규화해 공백/표기 차이로 인한 false negative를 줄입니다.
string_like_cols = df_norm.select_dtypes(include=["object", "string"]).columns.tolist()
for col in string_like_cols:
    df_norm[col] = df_norm[col].map(normalize_text)

add_fix_log(
    "normalize_text_columns",
    rows_before_norm,
    len(df_norm),
    cols_before_norm,
    df_norm.shape[1],
    f"normalized string columns: {string_like_cols}",
)

# 같은 강의계획서인지 판단할 핵심 비교 컬럼만 중복 기준 후보로 사용합니다.
dedup_key_candidates = [
    "course_name",
    "grade",
    "semester",
    "credit",
    "course_type",
    "learning_objective",
    "main_textbook",
    "sub_textbook",
    "reference_material",
    "prerequisite_material",
]
# provenance용/해석용 컬럼은 중복 판단 기준에서 의도적으로 제외합니다.
excluded_from_dedup = [
    "source_row_number",
    "college_name",
    "department_name",
    "is_target_department",
    "is_target_course",
]
dedup_key_cols = [col for col in dedup_key_candidates if col in df_norm.columns and col not in excluded_from_dedup]

rows_before_dup_check = len(df_norm)
cols_before_dup_check = df_norm.shape[1]

# 현재 기준으로 몇 개의 중복 그룹이 형성되는지 제거 전에 먼저 확인합니다.
duplicate_mask = df_norm.duplicated(subset=dedup_key_cols, keep=False) if dedup_key_cols else pd.Series(False, index=df_norm.index)
duplicate_row_count = int(df_norm.duplicated(subset=dedup_key_cols).sum()) if dedup_key_cols else 0

duplicate_group_summary_df = (
    df_norm.loc[duplicate_mask, dedup_key_cols]
    .value_counts(dropna=False)
    .reset_index(name="group_count")
    .sort_values("group_count", ascending=False)
    .head(50)
) if dedup_key_cols else pd.DataFrame(columns=["group_count"])

duplicate_group_count = int(duplicate_group_summary_df.shape[0])

duplicate_sample_cols = [
    "source_row_number",
    "college_name",
    "department_name",
    "course_name",
    "grade",
    "semester",
    "credit",
    "course_type",
    "learning_objective",
    "main_textbook",
    "prerequisite_material",
]
duplicate_sample_cols = [col for col in duplicate_sample_cols if col in df_norm.columns]
duplicate_sample_df = df_norm.loc[duplicate_mask, duplicate_sample_cols].copy().head(100)

print("[Duplicate Analysis] 전체 행 수")
print(len(df_norm))
print("\n[Duplicate Analysis] dedup_key_cols")
print(dedup_key_cols)
print("\n[Duplicate Analysis] dedup_key_cols 기준 중복 행 수")
print(duplicate_row_count)
print("\n[Duplicate Analysis] 중복 그룹 수")
print(duplicate_group_count)
print("\n[Duplicate Analysis] 중복 그룹별 개수 상위 50개")
display(duplicate_group_summary_df.head(50))
print("\n[Duplicate Analysis] 중복 샘플")
display(duplicate_sample_df.head(100))

add_fix_log(
    "duplicate_analysis",
    rows_before_dup_check,
    len(df_norm),
    cols_before_dup_check,
    df_norm.shape[1],
    f"dedup_key_cols={dedup_key_cols}; duplicate_rows={duplicate_row_count}; duplicate_groups={duplicate_group_count}",
)

rows_before_drop = len(df_norm)
cols_before_drop = df_norm.shape[1]
# keep="first" 기준으로 하나의 대표 행만 남기고 나머지 중복 강의계획서는 제거합니다.
df_final_fixed = df_norm.drop_duplicates(subset=dedup_key_cols, keep="first").copy() if dedup_key_cols else df_norm.copy()
rows_after_drop = len(df_final_fixed)
removed_rows = rows_before_drop - rows_after_drop
removed_ratio = round((removed_rows / rows_before_drop) * 100, 2) if rows_before_drop else 0.0

print("\n[Duplicate Drop] 중복 제거 전 행 수")
print(rows_before_drop)
print("\n[Duplicate Drop] 중복 제거 후 행 수")
print(rows_after_drop)
print("\n[Duplicate Drop] 제거된 행 수")
print(removed_rows)
print("\n[Duplicate Drop] 제거 비율")
print(f"{removed_ratio}%")

add_fix_log(
    "drop_syllabus_duplicates",
    rows_before_drop,
    rows_after_drop,
    cols_before_drop,
    df_final_fixed.shape[1],
    f"drop_duplicates with keep='first' using dedup_key_cols={dedup_key_cols}",
)

final_college_missing_count = int(df_final_fixed["college_name"].isna().sum()) if "college_name" in df_final_fixed.columns else len(df_final_fixed)
final_department_missing_count = int(df_final_fixed["department_name"].isna().sum()) if "department_name" in df_final_fixed.columns else len(df_final_fixed)

print("\n[Validation] df_final_fixed의 college_name 결측치 수")
print(final_college_missing_count)
print("\n[Validation] df_final_fixed의 department_name 결측치 수")
print(final_department_missing_count)

add_fix_log(
    "validate_metadata_columns",
    len(df_final_fixed),
    len(df_final_fixed),
    df_final_fixed.shape[1],
    df_final_fixed.shape[1],
    f"final missing college={final_college_missing_count}, final missing department={final_department_missing_count}",
)

check_source_ids = [17281, 17282]
check_before_df = df_work.loc[df_work["source_row_number"].isin(check_source_ids)].copy()
check_after_df = df_final_fixed.loc[df_final_fixed["source_row_number"].isin(check_source_ids)].copy()

print("\n[Validation] source_row_number 17281, 17282 - before")
display(check_before_df)
print("\n[Validation] source_row_number 17281, 17282 - after")
display(check_after_df)

# 메타데이터 복원 + 중복 고정까지 끝난 파일을 final 단계 산출물로 저장합니다.
output_path = Path("outputs/final/04_syllabus_metadata_restored_deduplicated.csv")
df_final_fixed.to_csv(output_path, index=False, encoding="utf-8-sig")

add_fix_log(
    "save_final_fixed_dataset",
    len(df_final_fixed),
    len(df_final_fixed),
    df_final_fixed.shape[1],
    df_final_fixed.shape[1],
    f"saved file to {output_path.name}",
)

eda_fix_log_df = pd.DataFrame(eda_fix_logs)

display(eda_fix_log_df)
display(df_work.head(5))
display(duplicate_group_summary_df.head(50))
display(duplicate_sample_df.head(100))
display(df_final_fixed.head(5))
display({
    "before_rows": check_before_df.to_dict(orient="records"),
    "after_rows": check_after_df.to_dict(orient="records"),
})

print(f"[Saved] {output_path.resolve()}")


## 9. 마지막 품질 정제: 결측치 기반 품질 필터링

마지막 단계는 남은 행들 중에서 추천 근거가 거의 없는 행만 제거하는 품질 정제입니다. 여기서는 모든 결측치를 동일하게 보지 않고, `learning_objective`, `main_textbook`, `course_name`, `course_type`, `college_name`, `department_name`처럼 추천과 해석에 직접 영향을 주는 컬럼은 더 엄격하게 평가합니다.

반대로 `sub_textbook`, `reference_material`, `prerequisite_material`은 보조 정보이므로, 이 값들만 비어 있다고 해서 과도하게 제거하지 않도록 설계합니다.


In [ ]:
# Purpose: 핵심 정보가 지나치게 부족한 행을 제거해 최종 추천용 품질 데이터셋을 만듭니다.
# Input: df_final_fixed
# Output: df_final_quality, df_removed_missing, quality_cleaning_log_df

# 9. 마지막 품질 정제: 결측치 기반 품질 필터링
# NOTE: df_final_fixed가 먼저 생성되어 있어야 합니다.

from pathlib import Path
import re
import html

quality_cleaning_logs = []

def add_quality_log(step, rows_before, rows_after, cols_before, cols_after, note):
    """마지막 품질 정제 단계에서 제거 규모와 저장 결과를 추적합니다."""
    quality_cleaning_logs.append({
        "step": step,
        "rows_before": rows_before,
        "rows_after": rows_after,
        "rows_removed": rows_before - rows_after,
        "cols_before": cols_before,
        "cols_after": cols_after,
        "cols_removed": cols_before - cols_after,
        "note": note,
    })

# 문자열 형태의 실질 결측값을 한 곳에서 관리해 품질 판단 기준을 일관되게 유지합니다.
generic_missing_tokens = {
    "", "-", "--", "/", "//", "/미정/", "미정", "추후 공지", "추후공고", "추후 공고",
    "해당없음", "해당 없음", "없다"
}
none_like_token = "없음"

def normalize_missing_value(value, column_name=""):
    """실질적 결측치를 통일하고, prerequisite_material의 없음은 실제 정보로 보존합니다."""
    if pd.isna(value):
        return ""
    text = html.unescape(str(value))
    text = text.replace("\u3000", " ")
    text = text.replace("\xa0", " ")
    text = text.replace("\u200b", " ")
    text = text.replace("\ufeff", " ")
    text = text.replace("\r", " ").replace("\n", " ").replace("\t", " ")
    text = re.sub(r"\s+", " ", text).strip()
    if text in generic_missing_tokens:
        return ""
    if text in {"없음", "없음."}:
        if column_name == "prerequisite_material":
            return none_like_token
        return ""
    return text

rows_before_prepare = len(df_final_fixed)
cols_before_prepare = df_final_fixed.shape[1]
# 최종 중복 제거 결과를 복사한 뒤, 행 단위 품질 점검 지표를 추가합니다.
df_quality_check = df_final_fixed.copy()
add_quality_log(
    "prepare_quality_check",
    rows_before_prepare,
    len(df_quality_check),
    cols_before_prepare,
    df_quality_check.shape[1],
    "copied df_final_fixed for quality filtering",
)

# 필수 식별 컬럼 / 핵심 추천 컬럼 / 보조 컬럼을 역할별로 분리합니다.
required_identifier_cols = [
    col for col in ["course_name", "course_type", "college_name", "department_name"]
    if col in df_quality_check.columns
]
core_content_cols = [
    col for col in ["learning_objective", "main_textbook"]
    if col in df_quality_check.columns
]
supporting_material_cols = [
    col for col in ["main_textbook", "sub_textbook", "reference_material", "prerequisite_material"]
    if col in df_quality_check.columns
]
quality_cols = [
    col for col in [
        "course_name", "grade", "semester", "credit", "course_type", "learning_objective",
        "main_textbook", "sub_textbook", "reference_material", "prerequisite_material",
        "college_name", "department_name"
    ] if col in df_quality_check.columns
]

# 각 컬럼을 결측 판단 규칙에 맞게 정규화한 캐시를 만들어 반복 계산을 줄입니다.
normalized_cache = {}
# 컬럼별 정규화 결과를 저장해 이후 결측 플래그/비율 계산에 재사용합니다.
for col in quality_cols:
    normalized_cache[col] = df_quality_check[col].map(lambda x, c=col: normalize_missing_value(x, c))

for col in quality_cols:
    df_quality_check[f"__norm__{col}"] = normalized_cache[col]

required_missing_flags = []
for col in required_identifier_cols:
    flag_col = f"__missing__{col}"
    df_quality_check[flag_col] = df_quality_check[f"__norm__{col}"] == ""
    required_missing_flags.append(flag_col)

core_missing_flags = []
for col in core_content_cols:
    flag_col = f"__missing__{col}"
    if flag_col not in df_quality_check.columns:
        df_quality_check[flag_col] = df_quality_check[f"__norm__{col}"] == ""
    core_missing_flags.append(flag_col)

quality_missing_flags = []
for col in quality_cols:
    flag_col = f"__quality_missing__{col}"
    df_quality_check[flag_col] = df_quality_check[f"__norm__{col}"] == ""
    quality_missing_flags.append(flag_col)

# 행 단위 결측 지표를 만들어 제거 기준을 수치적으로 판단할 수 있게 합니다.
df_quality_check["missing_required_count"] = df_quality_check[required_missing_flags].sum(axis=1) if required_missing_flags else 0
df_quality_check["missing_core_content_count"] = df_quality_check[core_missing_flags].sum(axis=1) if core_missing_flags else 0
df_quality_check["missing_quality_count"] = df_quality_check[quality_missing_flags].sum(axis=1) if quality_missing_flags else 0
df_quality_check["quality_missing_ratio"] = (
    df_quality_check["missing_quality_count"] / len(quality_cols)
) if quality_cols else 0.0

learning_norm_col = "__norm__learning_objective" if "learning_objective" in quality_cols else None
main_textbook_norm_col = "__norm__main_textbook" if "main_textbook" in quality_cols else None

df_quality_check["has_learning_objective"] = (
    df_quality_check[learning_norm_col] != ""
) if learning_norm_col else False
df_quality_check["has_main_textbook"] = (
    df_quality_check[main_textbook_norm_col] != ""
) if main_textbook_norm_col else False

# 주교재/부교재/참고자료/선수과목 중 하나라도 정보가 있는지 별도 플래그로 계산합니다.
supporting_norm_cols = [f"__norm__{col}" for col in supporting_material_cols if f"__norm__{col}" in df_quality_check.columns]
df_quality_check["has_any_supporting_material"] = False
if supporting_norm_cols:
    df_quality_check["has_any_supporting_material"] = df_quality_check[supporting_norm_cols].apply(
        lambda row: any(value not in {"", none_like_token} for value in row),
        axis=1,
    )
    if "prerequisite_material" in supporting_material_cols:
        prerequisite_norm_col = "__norm__prerequisite_material"
        df_quality_check["has_any_supporting_material"] = (
            df_quality_check["has_any_supporting_material"]
            | (df_quality_check[prerequisite_norm_col] == none_like_token)
        )

add_quality_log(
    "calculate_missing_indicators",
    len(df_quality_check),
    len(df_quality_check),
    cols_before_prepare,
    df_quality_check.shape[1],
    f"required={required_identifier_cols}, core={core_content_cols}, quality={quality_cols}",
)

# 실제 제거 조건을 boolean mask로 분리해두면 기준 변경과 검증이 쉬워집니다.
condition_required = df_quality_check["missing_required_count"] > 0
condition_missing_objective_and_textbook = (~df_quality_check["has_learning_objective"]) & (~df_quality_check["has_main_textbook"])
condition_high_missing_ratio = df_quality_check["quality_missing_ratio"] >= 0.6
condition_missing_objective_and_all_materials = (~df_quality_check["has_learning_objective"]) & (~df_quality_check["has_any_supporting_material"])

df_quality_check["removal_reason"] = ""
# 여러 제거 조건이 동시에 걸릴 수 있으므로 사유를 문자열로 누적합니다.
reason_map = {
    "missing_required_identifier": condition_required,
    "missing_learning_objective_and_main_textbook": condition_missing_objective_and_textbook,
    "high_row_missing_ratio": condition_high_missing_ratio,
    "missing_objective_and_all_materials": condition_missing_objective_and_all_materials,
}

for reason, mask in reason_map.items():
    df_quality_check.loc[mask, "removal_reason"] = df_quality_check.loc[mask, "removal_reason"].apply(
        lambda current, r=reason: r if current == "" else f"{current}; {r}"
    )

remove_mask = df_quality_check["removal_reason"] != ""
reason_summary_df = (
    df_quality_check.loc[remove_mask, "removal_reason"]
    .value_counts(dropna=False)
    .rename_axis("removal_reason")
    .reset_index(name="count")
)

add_quality_log(
    "identify_high_missing_rows",
    len(df_quality_check),
    len(df_quality_check),
    df_quality_check.shape[1],
    df_quality_check.shape[1],
    f"rows flagged for removal={int(remove_mask.sum())}",
)

rows_before_remove = len(df_quality_check)
cols_before_remove = df_quality_check.shape[1]
# 제거 대상과 최종 유지 대상을 분리해, 사후 검토와 저장을 모두 가능하게 합니다.
df_removed_missing = df_quality_check.loc[remove_mask].copy()
df_final_quality = df_quality_check.loc[~remove_mask].copy()

helper_cols_to_drop = [
    col for col in df_final_quality.columns
    if col.startswith("__norm__") or col.startswith("__missing__") or col.startswith("__quality_missing__")
]
df_removed_missing = df_removed_missing.drop(columns=helper_cols_to_drop, errors="ignore")
df_final_quality = df_final_quality.drop(columns=helper_cols_to_drop, errors="ignore")

rows_after_remove = len(df_final_quality)
removed_rows = rows_before_remove - rows_after_remove
removed_ratio = round((removed_rows / rows_before_remove) * 100, 2) if rows_before_remove else 0.0

add_quality_log(
    "remove_high_missing_rows",
    rows_before_remove,
    rows_after_remove,
    cols_before_remove,
    df_final_quality.shape[1],
    f"removed rows={removed_rows}, removed ratio={removed_ratio}%",
)

final_missing_rate_records = []
for col in quality_cols:
    normalized_series = df_final_quality[col].map(lambda x, c=col: normalize_missing_value(x, c))
    missing_ratio = round((normalized_series == "").mean() * 100, 2) if len(df_final_quality) else 0.0
    final_missing_rate_records.append({
        "column": col,
        "missing_count": int((normalized_series == "").sum()),
        "missing_ratio(%)": missing_ratio,
    })
final_missing_rate_df = pd.DataFrame(final_missing_rate_records).sort_values("missing_ratio(%)", ascending=False)

removed_sample_cols = [
    col for col in [
        "source_row_number", "course_name", "course_type", "college_name", "department_name",
        "learning_objective", "main_textbook", "sub_textbook", "reference_material",
        "prerequisite_material", "missing_required_count", "missing_core_content_count",
        "missing_quality_count", "quality_missing_ratio", "has_learning_objective",
        "has_main_textbook", "has_any_supporting_material", "removal_reason"
    ] if col in df_removed_missing.columns
]
removed_sample_df = df_removed_missing[removed_sample_cols].head(100)

print("[Quality] 제거 전 행 수")
print(rows_before_remove)
print("\n[Quality] 제거 후 행 수")
print(rows_after_remove)
print("\n[Quality] 제거된 행 수")
print(removed_rows)
print("\n[Quality] 제거 비율")
print(f"{removed_ratio}%")
print("\n[Quality] 제거 사유별 행 수")
display(reason_summary_df)
print("\n[Quality] 최종 데이터의 컬럼별 결측률")
display(final_missing_rate_df)
print("\n[Quality] 제거된 행 샘플 100개")
display(removed_sample_df)

# 최종 추천용 데이터셋과 제거 검토용 데이터셋을 각각 저장합니다.
quality_output_path = Path("outputs/final/05_syllabus_final_quality_filtered.csv")
removed_output_path = Path("outputs/final/05_syllabus_removed_by_quality_filter.csv")
df_final_quality.to_csv(quality_output_path, index=False, encoding="utf-8-sig")
df_removed_missing.to_csv(removed_output_path, index=False, encoding="utf-8-sig")

add_quality_log(
    "save_final_quality_dataset",
    len(df_final_quality),
    len(df_final_quality),
    df_final_quality.shape[1],
    df_final_quality.shape[1],
    f"saved final dataset to {quality_output_path.name} and removed rows to {removed_output_path.name}",
)

quality_cleaning_log_df = pd.DataFrame(quality_cleaning_logs)

display(quality_cleaning_log_df)
display(reason_summary_df)
display(final_missing_rate_df)
display(df_removed_missing.head(100))
display(df_final_quality.head(5))

print(f"[Saved] final quality dataset: {quality_output_path.resolve()}")
print(f"[Saved] removed rows dataset: {removed_output_path.resolve()}")


In [ ]:
# 10. RAG 청킹 전략 수립을 위한 텍스트 분석
# Purpose: 정제된 강의계획서 데이터의 주요 텍스트 필드를 분석해 RAG 청킹 기준을 정합니다.
# Input: df_final_quality 또는 저장된 최종/중간 산출물
# Output: rag_field_summary_df, rag_length_distribution_df, rag_chunking_strategy_df, rag_sample_df

import math
import re
import unicodedata
from pathlib import Path
from IPython.display import display
import pandas as pd
import numpy as np

pd.set_option("display.max_colwidth", 220)

def normalize_for_rag(value):
    """한글 자모 분리와 과도한 공백을 줄여 RAG 분석용 텍스트를 안정화합니다."""
    if pd.isna(value):
        return ""
    text = unicodedata.normalize("NFC", str(value))
    text = text.replace("\r\n", "\n").replace("\r", "\n")
    text = re.sub(r"[ \t]+", " ", text)
    text = re.sub(r"\n{3,}", "\n\n", text)
    return text.strip()

def load_syllabus_for_rag():
    for var_name in ["df_final_quality", "df_final_fixed", "df_final_eda", "df_cs_ee_clean"]:
        if var_name in globals():
            return globals()[var_name].copy(), f"memory: {var_name}"

    candidates = [
        Path("outputs/final/05_syllabus_final_quality_filtered.csv"),
        Path("outputs/final/04_syllabus_metadata_restored_deduplicated.csv"),
        Path("outputs/intermediate/03_syllabus_deduplicated.csv"),
        Path("outputs/intermediate/02_syllabus_domain_filtered.csv"),
    ]
    for path in candidates:
        if path.exists():
            return pd.read_csv(path, encoding="utf-8-sig"), str(path)
    raise FileNotFoundError("RAG 분석용 syllabus 최종/중간 산출물을 찾을 수 없습니다.")

rag_courses_df, rag_source = load_syllabus_for_rag()
rag_courses_df = rag_courses_df.copy()

text_fields = [
    col for col in [
        "course_name", "learning_objective", "main_textbook", "sub_textbook",
        "reference_material", "prerequisite_material", "course_type",
        "college_name", "department_name", "grade", "semester", "credit"
    ]
    if col in rag_courses_df.columns
]

for col in text_fields:
    rag_courses_df[col] = rag_courses_df[col].map(normalize_for_rag)

if "learning_objective" not in rag_courses_df.columns:
    raise KeyError("RAG 청킹 분석에 필요한 learning_objective 컬럼이 없습니다.")

content_fields = [
    col for col in [
        "course_name", "learning_objective", "main_textbook", "sub_textbook",
        "reference_material", "prerequisite_material"
    ]
    if col in rag_courses_df.columns
]
metadata_fields = [col for col in text_fields if col not in content_fields]

rag_courses_df["rag_text"] = rag_courses_df[content_fields].agg(
    lambda row: "\n".join(f"{col}: {row[col]}" for col in content_fields if row[col]),
    axis=1,
)

analysis_fields = [col for col in content_fields + ["rag_text"] if col in rag_courses_df.columns]

summary_rows = []
for col in analysis_fields:
    series = rag_courses_df[col].fillna("").map(normalize_for_rag)
    lengths = series.str.len()
    non_empty = lengths > 0
    summary_rows.append({
        "field": col,
        "rows": len(series),
        "non_empty_rows": int(non_empty.sum()),
        "empty_rows": int((~non_empty).sum()),
        "mean_chars": round(lengths[non_empty].mean(), 1) if non_empty.any() else 0,
        "median_chars": round(lengths[non_empty].median(), 1) if non_empty.any() else 0,
        "p75_chars": round(lengths[non_empty].quantile(0.75), 1) if non_empty.any() else 0,
        "p90_chars": round(lengths[non_empty].quantile(0.90), 1) if non_empty.any() else 0,
        "p95_chars": round(lengths[non_empty].quantile(0.95), 1) if non_empty.any() else 0,
        "max_chars": int(lengths.max()) if len(lengths) else 0,
    })

rag_field_summary_df = pd.DataFrame(summary_rows)

bins = [0, 100, 300, 500, 1000, 2000, 5000, np.inf]
labels = ["~100", "101~300", "301~500", "501~1000", "1001~2000", "2001~5000", "5000+"]
distribution_rows = []
for col in analysis_fields:
    lengths = rag_courses_df[col].fillna("").map(normalize_for_rag).str.len()
    bucket = pd.cut(lengths, bins=bins, labels=labels, include_lowest=True)
    counts = bucket.value_counts().sort_index()
    for label, count in counts.items():
        distribution_rows.append({"field": col, "length_bucket": str(label), "count": int(count)})

rag_length_distribution_df = pd.DataFrame(distribution_rows)

rag_text_lengths = rag_courses_df["rag_text"].str.len()
recommended_chunk_chars = int(min(1500, max(600, math.ceil(rag_text_lengths.quantile(0.90) / 100) * 100)))
recommended_overlap_chars = int(round(recommended_chunk_chars * 0.15 / 10) * 10)
long_record_threshold = recommended_chunk_chars * 1.5
long_record_count = int((rag_text_lengths > long_record_threshold).sum())

rag_chunking_strategy_df = pd.DataFrame([
    {
        "dataset": "curriculum_courses",
        "source": rag_source,
        "document_unit": "1 row = 1 course syllabus",
        "primary_text": "learning_objective + textbooks/material fields",
        "metadata_fields": ", ".join(metadata_fields),
        "recommended_chunk_chars": recommended_chunk_chars,
        "recommended_overlap_chars": recommended_overlap_chars,
        "split_rule": "강의 단위 유지, 교재/참고자료가 긴 행만 줄바꿈/문장 기준으로 분할",
        "long_record_threshold_chars": int(long_record_threshold),
        "long_record_count": long_record_count,
    }
])

rag_sample_columns = [col for col in ["course_name", "college_name", "department_name", "learning_objective", "main_textbook"] if col in rag_courses_df.columns]
rag_sample_df = (
    rag_courses_df.assign(rag_text_chars=rag_text_lengths)[rag_sample_columns + ["rag_text_chars"]]
    .sort_values("rag_text_chars", ascending=False)
    .head(5)
)

print("=" * 70)
print("RAG 청킹 전략 분석 - curriculum_courses")
print("=" * 70)
print(f"데이터 출처: {rag_source}")
print(f"행 수: {len(rag_courses_df):,}")
print("\n[필드별 텍스트 길이 요약]")
display(rag_field_summary_df)
print("\n[길이 구간 분포]")
display(rag_length_distribution_df.pivot(index="length_bucket", columns="field", values="count").fillna(0).astype(int))
print("\n[권장 청킹 전략]")
display(rag_chunking_strategy_df)
print("\n[긴 문서 샘플]")
display(rag_sample_df)
